# MLB Dataset Creation
Pulls game results, betting lines, pitcher logs, and team hitting logs from the MLB Stats API and merges them into a single modeling dataset.

In [1]:
# imports
import json
import time
import requests
import numpy as np
import pandas as pd

BASE_URL = "https://statsapi.mlb.com/api/v1"
SEASONS  = [2021, 2022, 2023, 2024, 2025]

## Game Results
Pulls every completed regular-season game for the target seasons. Rows with missing scores are dropped (postponements, incomplete games).

In [2]:
rows = []

for season in SEASONS:
    print(season)
    r = requests.get(
        f"{BASE_URL}/schedule",
        params={"sportId": 1, 
                "season": season, 
                "gameType": "R",  # regular season
                "hydrate": "probablePitcher"}
    )
    r.raise_for_status()

    for day in r.json().get("dates", []):
        for g in day.get("games", []):
            
            
            home_runs = g["teams"]["home"].get("score")
            away_runs = g["teams"]["away"].get("score")
            if home_runs is None or away_runs is None:
                continue
           
        
            home_pitcher = g["teams"]["home"].get("probablePitcher")
            away_pitcher = g["teams"]["away"].get("probablePitcher")

            rows.append({
                "season": season,
                "game_date": g["gameDate"][:10],
                "home_team": g["teams"]["home"]["team"]["name"],
                "away_team": g["teams"]["away"]["team"]["name"],
                "home_team_id": g["teams"]["home"]["team"]["id"],
                "away_team_id": g["teams"]["away"]["team"]["id"],
                "home_runs": int(home_runs),
                "away_runs": int(away_runs),
                "total_runs": int(home_runs) + int(away_runs),
                "mlb_gamePk": g["gamePk"],
                "home_pitcher_id":home_pitcher["id"] if home_pitcher else None,
                "away_pitcher_id":away_pitcher["id"] if away_pitcher else None,
                "home_pitcher_name":home_pitcher["fullName"] if home_pitcher else None,
                "away_pitcher_name":away_pitcher["fullName"] if away_pitcher else None,
            })

games_df = pd.DataFrame(rows)
games_df["game_date"] = pd.to_datetime(games_df["game_date"])
games_df["season"] = games_df["game_date"].dt.year
len(games_df)

2021
2022
2023
2024
2025


12171

In [3]:
games_df.head()

,season,game_date,home_team,away_team,home_team_id,away_team_id,home_runs,away_runs,total_runs,mlb_gamePk,home_pitcher_id,away_pitcher_id,home_pitcher_name,away_pitcher_name
0,2021,2021-04-01,New York Yankees,Toronto Blue Jays,147,141,2,3,5,634642,543037.0,547943.0,Gerrit Cole,Hyun Jin Ryu
1,2021,2021-04-01,Detroit Tigers,Cleveland Indians,116,114,3,2,5,634645,571510.0,669456.0,Matthew Boyd,Shane Bieber
2,2021,2021-04-01,Milwaukee Brewers,Minnesota Twins,158,142,6,5,11,634638,605540.0,628317.0,Brandon Woodruff,Kenta Maeda
3,2021,2021-04-01,Chicago Cubs,Pittsburgh Pirates,112,134,3,5,8,634634,543294.0,641771.0,Kyle Hendricks,Chad Kuhl
4,2021,2021-04-01,Philadelphia Phillies,Atlanta Braves,143,144,3,2,5,634622,605400.0,608331.0,Aaron Nola,Max Fried


## Team Rolling Stats
5-game and 10-game rolling averages for runs scored and runs allowed. Each row uses only games played *before* the current game to avoid leakage.

In [4]:
df = games_df

# Stack home and away into one row per team per game
home_rows = df[["season", "game_date", "home_team", "home_team_id", "home_runs", "away_runs"]]
home_rows = home_rows.rename(columns={"home_team": "team", "home_team_id": "team_id",
                                       "home_runs": "runs_for", "away_runs": "runs_against"})
home_rows["is_home"] = 1

away_rows = df[["season", "game_date", "away_team", "away_team_id", "away_runs", "home_runs"]]
away_rows = away_rows.rename(columns={"away_team": "team", "away_team_id": "team_id",
                                       "away_runs": "runs_for", "home_runs": "runs_against"})
away_rows["is_home"] = 0

team_games = pd.concat([home_rows, away_rows], ignore_index=True)
team_games = team_games.sort_values(["team_id", "game_date"]).reset_index(drop=True)

# Rolling averages shifted by 1 so current game is never included
team_games["rf_5"] = (
    team_games.groupby("team_id")["runs_for"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

team_games["ra_5"] = (
    team_games.groupby("team_id")["runs_against"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))

team_games["rf_10"] = (
    team_games.groupby("team_id")["runs_for"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()))

team_games["ra_10"] = (
    team_games.groupby("team_id")["runs_against"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean()))



feats = team_games[["season", "game_date", "team_id", "is_home", "rf_5", "ra_5", "rf_10", "ra_10"]]

home_feat = feats[feats["is_home"] == 1].rename(columns={
    "team_id": "home_team_id", 
    "rf_5": "home_rf_5", 
    "ra_5": "home_ra_5",
    "rf_10": "home_rf_10", 
    "ra_10": "home_ra_10"
})[["season", "game_date", "home_team_id", "home_rf_5", "home_ra_5", "home_rf_10", "home_ra_10"]]

away_feat = feats[feats["is_home"] == 0].rename(columns={
    "team_id": "away_team_id", 
    "rf_5": "away_rf_5", 
    "ra_5": "away_ra_5",
    "rf_10": "away_rf_10", 
    "ra_10": "away_ra_10",
})[["season", "game_date", "away_team_id", "away_rf_5", "away_ra_5", "away_rf_10", "away_ra_10"]]

team_rolling_df = (
    df.merge(
        home_feat,on=["season", "game_date", "home_team_id"], how="left").merge(
        away_feat,on=["season", "game_date", "away_team_id"], how="left"))

team_rolling_df.shape

(22188, 22)

## Betting Lines
Loads DraftKings totals from a local JSON file and merges them onto the game results by date and home team.

In [5]:
with open("mlb_odds_dataset.json", "r") as file:
    data = json.load(file)

rows = []
for date, games in data.items():
    for game in games:
        info = game["gameView"]
        dk_totals = next(
            (bk for bk in game["odds"].get("totals", []) if bk["sportsbook"] == "draftkings"),
            None
        )
        if dk_totals is None:
            continue

        current = dk_totals.get("currentLine", {})
        rows.append({
            "date":date,
            "away_team":info["awayTeam"]["fullName"],
            "home_team":info["homeTeam"]["fullName"],
            "venue":info.get("venueName"),
            "total_line":current.get("total"),
            "over_odds":current.get("overOdds"),
            "under_odds":current.get("underOdds"),
        })

betting_df = pd.DataFrame(rows)
betting_df["date"] = pd.to_datetime(betting_df["date"])

betting_df.head()

,date,away_team,home_team,venue,total_line,over_odds,under_odds
0,2021-04-01,Toronto Blue Jays,New York Yankees,Yankee Stadium,8.0,-105,-117
1,2021-04-01,Cleveland Guardians,Detroit Tigers,Comerica Park,7.5,-108,-113
2,2021-04-01,Minnesota Twins,Milwaukee Brewers,American Family Field,7.5,100,-121
3,2021-04-01,Pittsburgh Pirates,Chicago Cubs,Wrigley Field,7.0,100,-121
4,2021-04-01,Atlanta Braves,Philadelphia Phillies,Citizens Bank Park,7.0,-125,104


In [6]:
# Cleaning betting data team names

# Cleveland was renamed from Indians to Guardians mid-dataset
betting_df["home_team"] = betting_df["home_team"].replace({
    "Cleveland Guardians":   "Cleveland Indians",
    "Athletics Athletics": "Oakland Athletics",
})
betting_df["away_team"] = betting_df["away_team"].replace({
    "Cleveland Guardians":   "Cleveland Indians",
    "Athletics Athletics": "Oakland Athletics",
})

all_star_teams = {"National League All-Stars", "American League All-Stars"}
betting_df = betting_df[~betting_df["home_team"].isin(all_star_teams)].reset_index(drop=True)
betting_df = betting_df[~betting_df["away_team"].isin(all_star_teams)].reset_index(drop=True)

In [7]:
betting_df.isna().sum()

date          0
away_team     0
home_team     0
venue         0
total_line    0
over_odds     0
under_odds    0
dtype: int64

### Merge Lines onto Games

In [8]:
games_with_lines = games_df.merge(
    betting_df[["date", "home_team", "venue", "total_line", "over_odds", "under_odds"]],
    left_on=["game_date", "home_team"],
    right_on=["date", "home_team"],
    how="left"
).drop(columns=["date"])

games_with_lines = games_with_lines.dropna()
games_with_lines.head()

,season,game_date,home_team,away_team,home_team_id,away_team_id,home_runs,away_runs,total_runs,mlb_gamePk,home_pitcher_id,away_pitcher_id,home_pitcher_name,away_pitcher_name,venue,total_line,over_odds,under_odds
0,2021,2021-04-01,New York Yankees,Toronto Blue Jays,147,141,2,3,5,634642,543037.0,547943.0,Gerrit Cole,Hyun Jin Ryu,Yankee Stadium,8.0,-105.0,-117.0
1,2021,2021-04-01,Detroit Tigers,Cleveland Indians,116,114,3,2,5,634645,571510.0,669456.0,Matthew Boyd,Shane Bieber,Comerica Park,7.5,-108.0,-113.0
2,2021,2021-04-01,Milwaukee Brewers,Minnesota Twins,158,142,6,5,11,634638,605540.0,628317.0,Brandon Woodruff,Kenta Maeda,American Family Field,7.5,100.0,-121.0
3,2021,2021-04-01,Chicago Cubs,Pittsburgh Pirates,112,134,3,5,8,634634,543294.0,641771.0,Kyle Hendricks,Chad Kuhl,Wrigley Field,7.0,100.0,-121.0
4,2021,2021-04-01,Philadelphia Phillies,Atlanta Braves,143,144,3,2,5,634622,605400.0,608331.0,Aaron Nola,Max Fried,Citizens Bank Park,7.0,-125.0,104.0


## Starting Pitcher Stats
Fetches game-by-game pitching logs for every starter in the dataset. Prior cumulative stats are shifted by one game so nothing from the current start leaks in. For a pitcher's first game of the season, the previous season's full stats are used as a fallback. True rookies with no prior data default to 0.

In [9]:
games_df["home_pitcher_id"] = pd.to_numeric(games_df["home_pitcher_id"], errors="coerce")
games_df["away_pitcher_id"] = pd.to_numeric(games_df["away_pitcher_id"], errors="coerce")

all_pitcher_ids = pd.concat([
    games_df["home_pitcher_id"], games_df["away_pitcher_id"]
    ]).dropna().astype(int).unique()

all_seasons = games_df["season"].unique()
all_seasons_with_prior = np.unique(np.concatenate([all_seasons, all_seasons - 1]))

fetch_pairs = pd.DataFrame([(pid, szn)
    for pid in all_pitcher_ids
    for szn in all_seasons_with_prior], 
    columns=["pitcher_id", "season"])

# Fetch pitching game logs
all_rows = []
for row in fetch_pairs.itertuples(index=False):
    pitcher_id = int(row.pitcher_id)
    season = int(row.season)
    
    try:
        r = requests.get(
            f"{BASE_URL}/people/{pitcher_id}/stats",
            params={"stats": "gameLog", "group": "pitching", "season": season},
            timeout=30)
        r.raise_for_status()
        splits = r.json().get("stats", [{}])[0].get("splits", [])
    except Exception:
        splits = []

    for s in splits:
        stat   = s.get("stat", {})
        ip_raw = str(stat.get("inningsPitched", "0")).strip()
        if "." in ip_raw:
            whole, frac = ip_raw.split(".")
            ip_outs = int(whole) * 3 + int(frac)
        else:
            ip_outs = int(float(ip_raw)) * 3 if ip_raw else 0

        all_rows.append({
            "pitcher_id": pitcher_id,
            "season": season,
            "game_date": pd.to_datetime(s.get("date")),
            "gamePk": s.get("game", {}).get("gamePk"),
            "outs_pitched": ip_outs,
            "strikeouts":stat.get("strikeOuts", 0),
            "hits": stat.get("hits", 0),
            "earnedruns": stat.get("earnedRuns", 0),
            "homeruns": stat.get("homeRuns", 0),
            "baseonballs":stat.get("baseOnBalls", 0),
        })
    time.sleep(0.02)

raw_logs = pd.DataFrame(all_rows)

In [10]:
raw_logs.head()

,pitcher_id,season,game_date,gamePk,outs_pitched,strikeouts,hits,earnedruns,homeruns,baseonballs
0,543037,2020,2020-07-23,630851,15,5,1,1,1,1
1,543037,2020,2020-07-29,630495,20,7,4,3,1,2
2,543037,2020,2020-08-03,631219,18,4,5,1,1,1
3,543037,2020,2020-08-08,630962,14,10,6,3,1,1
4,543037,2020,2020-08-14,631196,21,8,4,1,1,0


In [11]:
# Cumulative prior stats; shift by 1 game so current start is not included
stat_cols = ["strikeouts", "hits", "earnedruns", "homeruns", "baseonballs", "outs_pitched"]
logs = raw_logs.copy()

for col in stat_cols:
    logs[col] = pd.to_numeric(logs[col], errors="coerce").fillna(0) # True rookie gets 0

logs = logs.sort_values(["pitcher_id", "season", "game_date", "gamePk"]).reset_index(drop=True)

for col in stat_cols:
    logs[f"cum_{col}"] = (
        logs.groupby(["pitcher_id", "season"])[col].transform(
            lambda x: x.cumsum().shift(1).fillna(0)))

logs["k_per9"] = np.where(logs["cum_outs_pitched"] > 0, 27 * logs["cum_strikeouts"] / logs["cum_outs_pitched"], np.nan)
logs["era"] = np.where(logs["cum_outs_pitched"] > 0, 27 * logs["cum_earnedruns"] / logs["cum_outs_pitched"], np.nan)
logs["hits_per9"] = np.where(logs["cum_outs_pitched"] > 0, 27 * logs["cum_hits"] / logs["cum_outs_pitched"], np.nan)
logs["whip"]  = np.where(logs["cum_outs_pitched"] > 0,  3 * (logs["cum_hits"] + logs["cum_baseonballs"]) / logs["cum_outs_pitched"], np.nan)
logs["hr_per9"] = np.where(logs["cum_outs_pitched"] > 0, 27 * logs["cum_homeruns"]/ logs["cum_outs_pitched"], np.nan)

prior_df = logs[["pitcher_id", "season", "game_date", "gamePk",
                "k_per9", "era", "hits_per9", "whip", "hr_per9"]]

# Deduplicate to handle doubleheaders (keep last game of the day)
prior_df = prior_df.sort_values("game_date").drop_duplicates(
    subset=["pitcher_id", "season", "game_date"], keep="last")


# Previous-season fallback for first game of the year
season_totals = logs.groupby(["pitcher_id", "season"])[stat_cols].sum().reset_index()
season_totals["fb_k_per9"]= np.where(season_totals["outs_pitched"] > 0, 27 * season_totals["strikeouts"] / season_totals["outs_pitched"], 0)
season_totals["fb_era"] = np.where(season_totals["outs_pitched"] > 0, 27 * season_totals["earnedruns"] / season_totals["outs_pitched"], 0)
season_totals["fb_hits_per9"] = np.where(season_totals["outs_pitched"] > 0, 27 * season_totals["hits"] / season_totals["outs_pitched"], 0)
season_totals["fb_whip"] = np.where(season_totals["outs_pitched"] > 0,  3 * (season_totals["hits"] + season_totals["baseonballs"]) / season_totals["outs_pitched"], 0)
season_totals["fb_hr_per9"] = np.where(season_totals["outs_pitched"] > 0, 27 * season_totals["homeruns"] / season_totals["outs_pitched"], 0)

fallback = season_totals[["pitcher_id", "season", "fb_k_per9", "fb_era", "fb_hits_per9", "fb_whip", "fb_hr_per9"]].copy()
fallback["season"] = fallback["season"] + 1


# Merge onto games
games_with_pitcher_stats = games_df.copy()
rate_cols = ["k_per9", "era", "hits_per9", "whip", "hr_per9"]
fb_cols   = ["fb_k_per9",    "fb_era",    "fb_hits_per9",    "fb_whip",    "fb_hr_per9"]

for side in ["home", "away"]:
    pid_col = f"{side}_pitcher_id"

    games_with_pitcher_stats = games_with_pitcher_stats.merge(
        prior_df.rename(columns={"pitcher_id": pid_col, **{c: f"{side}_{c}" for c in rate_cols}}),
        on=[pid_col, "season", "game_date"], how="left"
    )
    games_with_pitcher_stats = games_with_pitcher_stats.merge(
        fallback.rename(columns={"pitcher_id": pid_col, **{c: f"{side}_{c}" for c in fb_cols}}),
        on=[pid_col, "season"], how="left"
    )
    for stat in ["k_per9", "era", "hits_per9", "whip", "hr_per9"]:
        games_with_pitcher_stats[f"{side}_{stat}"] = (
            games_with_pitcher_stats[f"{side}_{stat}"]
            .fillna(games_with_pitcher_stats[f"{side}_fb_{stat}"])
            .fillna(0)
        )
    games_with_pitcher_stats = games_with_pitcher_stats.drop(
        columns=[f"{side}_fb_{s}" for s in ["k_per9", "era", "hits_per9", "whip", "hr_per9"]])

len(games_with_pitcher_stats)

12171

## Team Hitting Stats
Same leakage-free approach as pitching — cumulative season stats shifted by one game, with prior-season fallback for Opening Day.

In [12]:
all_team_ids = pd.concat([
    games_df["home_team_id"], games_df["away_team_id"]
]).dropna().astype(int).unique()

team_fetch_pairs = pd.DataFrame([
    (tid, szn)
    for tid in all_team_ids
    for szn in all_seasons_with_prior],
    columns=["team_id", "season"])

# Fetch hitting game logs
hitting_rows = []
for row in team_fetch_pairs.itertuples(index=False):
    team_id, season = int(row.team_id), int(row.season)
    try:
        r = requests.get(
            f"{BASE_URL}/teams/{team_id}/stats",
            params={"stats": "gameLog", "group": "hitting", "season": season},
            timeout=30
        )
        r.raise_for_status()
        splits = r.json().get("stats", [{}])[0].get("splits", [])
    except Exception:
        splits = []

    for s in splits:
        stat = s.get("stat", {})
        hitting_rows.append({
            "team_id": team_id,
            "season": season,
            "game_date": pd.to_datetime(s.get("date")),
            "gamePk": s.get("game", {}).get("gamePk"),
            "runs": stat.get("runs", 0),
            "hits": stat.get("hits", 0),
            "doubles": stat.get("doubles", 0),
            "triples": stat.get("triples", 0),
            "homeruns": stat.get("homeRuns", 0),
            "strikeouts": stat.get("strikeOuts", 0),
            "baseonballs": stat.get("baseOnBalls", 0),
            "atbats": stat.get("atBats", 0),
        })
    time.sleep(0.02)

raw_hitting_logs = pd.DataFrame(hitting_rows)

In [13]:
# Cumulative prior stats
hit_count_cols = ["runs", "hits", "doubles", "triples", "homeruns", "strikeouts", "baseonballs", "atbats"]
hlogs = raw_hitting_logs.copy()
for col in hit_count_cols:
    hlogs[col] = pd.to_numeric(hlogs[col], errors="coerce").fillna(0)

hlogs = hlogs.sort_values(["team_id", "season", "game_date", "gamePk"]).reset_index(drop=True)

for col in hit_count_cols:
    hlogs[f"cum_{col}"] = (
        hlogs.groupby(["team_id", "season"])[col].transform(
            lambda x: x.cumsum().shift(1).fillna(0)))

games_played = hlogs.groupby(["team_id", "season"]).cumcount()

hlogs["avg"] = np.where(hlogs["cum_atbats"] > 0, hlogs["cum_hits"] / hlogs["cum_atbats"], np.nan)
hlogs["obp"] = np.where(hlogs["cum_atbats"] > 0, (hlogs["cum_hits"] + hlogs["cum_baseonballs"]) / (hlogs["cum_atbats"] + hlogs["cum_baseonballs"]), np.nan)
hlogs["slg"] = np.where(hlogs["cum_atbats"] > 0, (hlogs["cum_hits"] + hlogs["cum_doubles"] + 2 * hlogs["cum_triples"] + 3 * hlogs["cum_homeruns"]) / hlogs["cum_atbats"], np.nan)
hlogs["ops"] = hlogs["obp"] + hlogs["slg"]
hlogs["k_rate"] = np.where(hlogs["cum_atbats"] > 0, hlogs["cum_strikeouts"]  / hlogs["cum_atbats"], np.nan)
hlogs["bb_rate"] = np.where(hlogs["cum_atbats"] > 0, hlogs["cum_baseonballs"] / hlogs["cum_atbats"], np.nan)
hlogs["hr_rate"] = np.where(hlogs["cum_atbats"] > 0, hlogs["cum_homeruns"]    / hlogs["cum_atbats"], np.nan)
hlogs["rpg"] = np.where(games_played > 0, hlogs["cum_runs"] / games_played, np.nan)


hit_rate_cols = ["avg", "obp", "slg", "ops", "k_rate", "bb_rate", "hr_rate", "rpg"]

prior_hitting_df = hlogs[["team_id", "season", "game_date"] + hit_rate_cols].copy()
prior_hitting_df = prior_hitting_df.sort_values("game_date").drop_duplicates(
    subset=["team_id", "season", "game_date"], keep="last")


# Previous-season fallback
hit_season_totals = hlogs.groupby(["team_id", "season"])[hit_count_cols].sum().reset_index()
games_per_season  = hlogs.groupby(["team_id", "season"]).size().reset_index(name="games_played")
hit_season_totals = hit_season_totals.merge(games_per_season, on=["team_id", "season"])

hit_season_totals["fb_avg"] = np.where(hit_season_totals["atbats"] > 0, hit_season_totals["hits"] / hit_season_totals["atbats"], 0)
hit_season_totals["fb_obp"] = np.where(hit_season_totals["atbats"] > 0, (hit_season_totals["hits"] + hit_season_totals["baseonballs"]) / (hit_season_totals["atbats"] + hit_season_totals["baseonballs"]), 0)
hit_season_totals["fb_slg"] = np.where(hit_season_totals["atbats"] > 0, (hit_season_totals["hits"] + hit_season_totals["doubles"] + 2 * hit_season_totals["triples"] + 3 * hit_season_totals["homeruns"]) / hit_season_totals["atbats"], 0)
hit_season_totals["fb_ops"] = hit_season_totals["fb_obp"] + hit_season_totals["fb_slg"]
hit_season_totals["fb_k_rate"] = np.where(hit_season_totals["atbats"] > 0, hit_season_totals["strikeouts"]  / hit_season_totals["atbats"], 0)
hit_season_totals["fb_bb_rate"] = np.where(hit_season_totals["atbats"] > 0, hit_season_totals["baseonballs"] / hit_season_totals["atbats"], 0)
hit_season_totals["fb_hr_rate"] = np.where(hit_season_totals["atbats"] > 0, hit_season_totals["homeruns"]    / hit_season_totals["atbats"], 0)
hit_season_totals["fb_rpg"] = np.where(hit_season_totals["games_played"] > 0, hit_season_totals["runs"] / hit_season_totals["games_played"], 0)

hit_fb_cols  = ["fb_avg", "fb_obp", "fb_slg", "fb_ops", "fb_k_rate", "fb_bb_rate", "fb_hr_rate", "fb_rpg"]
hit_fallback = hit_season_totals[["team_id", "season"] + hit_fb_cols].copy()
hit_fallback["season"] = hit_fallback["season"] + 1

# Merge onto games
games_with_hitting_stats = games_df.copy()
for side in ["home", "away"]:
    tid_col = f"{side}_team_id"

    games_with_hitting_stats = games_with_hitting_stats.merge(
        prior_hitting_df.rename(columns={"team_id": tid_col, **{c: f"{side}_{c}" for c in hit_rate_cols}}),
        on=[tid_col, "season", "game_date"], how="left")
    
    games_with_hitting_stats = games_with_hitting_stats.merge(
        hit_fallback.rename(columns={"team_id": tid_col, **{c: f"{side}_{c}" for c in hit_fb_cols}}),
        on=[tid_col, "season"], how="left")
    
    for stat in ["avg", "obp", "slg", "ops", "k_rate", "bb_rate", "hr_rate", "rpg"]:
        games_with_hitting_stats[f"{side}_{stat}"] = (
            games_with_hitting_stats[f"{side}_{stat}"]
            .fillna(games_with_hitting_stats[f"{side}_fb_{stat}"])
            .fillna(0))
        
    games_with_hitting_stats = games_with_hitting_stats.drop(
        columns=[f"{side}_fb_{s}" for s in ["avg", "obp", "slg", "ops", "k_rate", "bb_rate", "hr_rate", "rpg"]])

len(games_with_hitting_stats)

12171

## Build model_df
Merges all feature sets into a single row-per-game dataframe. Duplicate columns are dropped before merging to avoid `_x`/`_y` suffixes.

In [14]:
JOIN_KEYS = ["season", "game_date", "home_team_id", "away_team_id", "total_runs"]

DROP_FROM_ROLLING = ["home_team", "away_team", "home_runs", "away_runs",
                     "mlb_gamePk", "home_pitcher_id", "away_pitcher_id",
                     "home_pitcher_name", "away_pitcher_name"]

DROP_FROM_PITCHER = ["home_team", "away_team", "home_runs", "away_runs",
                     "mlb_gamePk", "home_pitcher_name", "away_pitcher_name"]

DROP_FROM_HITTING = ["home_team", "away_team", "home_runs", "away_runs",
                     "mlb_gamePk", "home_pitcher_id", "away_pitcher_id",
                     "home_pitcher_name", "away_pitcher_name"]

model_df = (
    games_with_lines
    .merge(team_rolling_df.drop(columns=DROP_FROM_ROLLING),
           on=JOIN_KEYS, how="left")
    .merge(games_with_pitcher_stats.drop(columns=DROP_FROM_PITCHER),
           on=JOIN_KEYS + ["home_pitcher_id", "away_pitcher_id"], how="left")
    .merge(games_with_hitting_stats.drop(columns=DROP_FROM_HITTING, errors="ignore"),
           on=JOIN_KEYS, how="left")
)

model_df = model_df.drop_duplicates(subset=JOIN_KEYS)

### Ballpark factors
##### (was not included in project proposal)

Each ballpark is different, and this may play a massive role in total runs scored. For this, I will be using a calculation based baseball reference's Park Adjustment metric. Found more details here: https://www.baseball-reference.com/about/parkadjust.shtml

Since there is no scrapable API, I have to make a list to map it. There are a total of 45 venues in my database. Some of which have no oark factor due to limited data. For these parks, I used 100, which is the neutral number

##### PLEASE NOTE I USED AI TO CREATE THIS LIST. #####
This was done by pasting a screenshot of the metrics into ChatGPT and asking it to create a python list.

In [15]:
park_factors = {
    'Yankee Stadium': 102,
    'Comerica Park': 98,
    'American Family Field': 101,
    'Wrigley Field': 102,
    'Citizens Bank Park': 104,
    'Coors Field': 112,
    'PETCO Park': 94,
    'Great American Ball Park': 105,
    'Kauffman Stadium': 97,
    'loanDepot park': 102,
    'Angel Stadium of Anaheim': 99,
    'Oakland Coliseum': 96,
    'T-Mobile Park': 96,
    'Fenway Park': 104,
    'Globe Life Field': 103,
    'Progressive Field': 99,
    'Nationals Park': 97,
    'Citi Field': 98,
    'PNC Park': 96,
    'Oriole Park at Camden Yards': 96,
    'Target Field': 98,
    'Guaranteed Rate Field': 101,
    'Busch Stadium': 97,
    'Minute Maid Park': 101,
    'Tropicana Field': 97,
    'Dodger Stadium': 97,
    'Oracle Park': 95,
    'Truist Park': 103,
    'Chase Field': 104,
    'Rogers Centre': 101,

    # this are deplicates from above (stadiums changed names)
    'Rate Field': 101,
    'Angel Stadium': 99,
    'Daikin Park':101}

model_df['park_factor'] = model_df['venue'].map(park_factors).fillna(100)

In [16]:
model_df.drop(
        ['home_team_id','away_team_id','mlb_gamePk','home_pitcher_id','away_pitcher_id','gamePk_x','gamePk_y'],
    axis=1, inplace=True)

In [17]:
model_df.dropna(inplace=True)

In [18]:
model_df.to_csv('model.csv', index=False)